In [28]:
import os, pickle, re
import numpy as np
import pandas as pd
import json

from physics.simulation import mcfm, msq
from physics.hstar import eft
from physics.variables import eft_terms, c6_degree, ct_degree, cg_degree
from physics.constants import c6_to_cH, cg_to_cHG, ct_to_ctH
from nsbi import carl, taylr

import torch
from torch.utils.data import TensorDataset, DataLoader
import lightning as L

In [29]:
torch.set_float32_matmul_precision('high')

import matplotlib, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "lualatex",
    "text.usetex": True,
    "pgf.rcfonts": False,
    "pgf.preamble": r"\usepackage{amssymb}",
})

In [30]:
run_dir_4l = 'run/h4l'
run_dir_2l2v = 'run/h2l2v'

In [31]:
(events_gg_4l_train, events_gg_4l), scaler_X_taylr_4l, models_taylr_4l, scalers_y_taylr_4l = taylr.utils.load_results(run_dir_4l, eft_terms)
(events_gg_2l2v_train, events_gg_2l2v), scaler_X_taylr_2l2v, models_taylr_2l2v, scalers_y_taylr_2l2v = taylr.utils.load_results(run_dir_2l2v, eft_terms)

In [32]:
(events_qq_4l_train, events_qq_4l), _, scaler_carl_4l, model_carl_4l = carl.utils.load_results(run_dir_4l)
(events_qq_2l2v_train, events_qq_2l2v), _, scaler_carl_2l2v, model_carl_2l2v = carl.utils.load_results(run_dir_2l2v)

In [33]:
lumi = 3000.

c6_linspace = [-25,25,121]
c6_space = np.linspace(*c6_linspace)
c6_sm = 0.0
i_c6_sm = np.where(np.isclose(c6_space,c6_sm))[0][0]
c6_asimov = 0.0
i_c6_asimov = np.where(np.isclose(c6_space,c6_asimov))[0][0]

ct_linspace = [-1.25,1.25,121]
ct_space = np.linspace(*ct_linspace)
ct_sm = 0.0
i_ct_sm = np.where(np.isclose(ct_space, ct_sm))[0][0]
ct_asimov = 0.0
i_ct_asimov = np.where(np.isclose(ct_space, ct_asimov))[0][0]

cg_linspace = [-2.5,2.5,121]
cg_space = np.linspace(*cg_linspace)
cg_sm = 0.0
i_cg_sm = np.where(np.isclose(cg_space, cg_sm))[0][0]
cg_asimov = 0.0
i_cg_asimov = np.where(np.isclose(cg_space, cg_asimov))[0][0]

In [34]:
class ZeroWeightFilter():
    def __init__(self):
        pass
    def __call__(self, kinematics, components, weights, probabilities):
        return np.where(weights != 0)

sample_size = 200_000

events_gg_4l, events_gg_2l2v = events_gg_4l.sample(sample_size), events_gg_2l2v.filter(ZeroWeightFilter()).sample(sample_size)
events_qq_4l, events_qq_2l2v = events_qq_4l.sample(sample_size), events_qq_2l2v.filter(ZeroWeightFilter()).sample(sample_size)

w_gg_4l_sm, w_gg_2l2v_sm = events_gg_4l.weights.to_numpy(), events_gg_2l2v.weights.to_numpy()
w_qq_4l_sm, w_qq_2l2v_sm = events_qq_4l.weights.to_numpy(), events_qq_2l2v.weights.to_numpy()

sigma_gg_4l_sm = np.sum(w_gg_4l_sm)
sigma_gg_2l2v_sm = np.sum(w_gg_2l2v_sm)
sigma_qq_4l_sm = np.sum(w_qq_4l_sm)
sigma_qq_2l2v_sm = np.sum(w_qq_2l2v_sm)

events_4l = mcfm.stack(events_gg_4l, events_qq_4l)
events_2l2v = mcfm.stack(events_gg_2l2v, events_qq_2l2v)
sigma_4l_sm = events_4l.weights.sum()
sigma_2l2v_sm = events_2l2v.weights.sum()

events_gg = mcfm.stack(events_gg_4l, events_gg_2l2v)
events_qq = mcfm.stack(events_qq_4l, events_qq_2l2v)
sigma_gg_sm = events_gg.weights.sum()
sigma_qq = events_qq.weights.sum()

In [35]:
features_4l = ['l1_pt', 'l1_eta', 'l1_phi', 'l1_energy', 'l2_pt', 'l2_eta', 'l2_phi', 'l2_energy', 'l3_pt', 'l3_eta', 'l3_phi', 'l3_energy', 'l4_pt', 'l4_eta', 'l4_phi', 'l4_energy']
features_2l2v = ["l1_pt", "l1_eta", "l1_phi", "l1_energy", "l2_pt", "l2_eta", "l2_phi", "l2_energy", "met", "met_phi"]

batch_size = 2048

In [36]:
r_4l = carl.utils.get_likelihood_ratio(events_4l, features_4l, scaler_carl_4l, model_carl_4l)
r_2l2v = carl.utils.get_likelihood_ratio(events_2l2v, features_2l2v, scaler_carl_2l2v, model_carl_2l2v)
r = torch.cat((r_4l, r_2l2v))

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

In [37]:
coeffs_4l = taylr.utils.get_coefficients(events_4l, features_4l, scaler_X_taylr_4l, models_taylr_4l, scalers_y_taylr_4l)
coeffs_2l2v = taylr.utils.get_coefficients(events_2l2v, features_2l2v, scaler_X_taylr_2l2v, models_taylr_2l2v, scalers_y_taylr_2l2v)
coeffs = torch.cat((coeffs_4l,coeffs_2l2v),dim=0)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


...

Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

In [38]:
coeffs_gg_4l = taylr.utils.get_coefficients(events_gg_4l, features_4l, scaler_X_taylr_4l, models_taylr_4l, scalers_y_taylr_4l)
coeffs_gg_2l2v = taylr.utils.get_coefficients(events_gg_2l2v, features_2l2v, scaler_X_taylr_2l2v, models_taylr_2l2v, scalers_y_taylr_2l2v)
coeffs_gg = torch.cat((coeffs_gg_4l,coeffs_gg_2l2v),dim=0)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


...

Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

In [85]:
cx_name = 'c6'
cy_name = 'ct'

cx_linspace = c6_linspace
cy_linspace = ct_linspace

cx_space = c6_space * c6_to_cH
cy_space = ct_space * ct_to_ctH

cx_min, cx_max = -60, 50
cy_min, cy_max = -12, 24

cx_label = '$C_H$'
cy_label = '$C_{tH}$'

cxyz_space = (c6_space, ct_space, None)
cxyz_asimov = (c6_asimov, ct_asimov, None)

In [111]:
cx_name = 'ct'
cy_name = 'cg'

cx_linspace = ct_linspace
cy_linspace = cg_linspace

cx_space = ct_space * ct_to_ctH
cy_space = cg_space * cg_to_cHG

cx_min, cx_max = -10, 15
cy_min, cy_max = -0.15, 0.25

cx_label = '$C_{tH}$'
cy_label = '$C_{HG}$'

cxyz_space = (None, ct_space, cg_space)
cxyz_asimov = (None, ct_asimov, cg_asimov)

In [55]:
cx_name = 'c6'
cy_name = 'cg'

cx_linspace = c6_linspace
cy_linspace = cg_linspace

cx_space = c6_space * c6_to_cH
cy_space = cg_space * cg_to_cHG

cx_min, cx_max = -60, 50
cy_min, cy_max = -0.15, 0.25

cx_label = '$C_{H}$'
cy_label = '$C_{HG}$'

cxyz_space = (c6_space, None, cg_space)
cxyz_asimov = (c6_asimov, None, cg_asimov)

In [112]:
# 4l

w_gg_4l_sm = torch.tensor(events_gg_4l.weights.to_numpy())
sigma_gg_4l_sm = torch.sum(w_gg_4l_sm)

w_gg_4l_bsm = w_gg_4l_sm[:,None,None] * torch.tensor(eft.msq_eft_over_sm(coeffs_gg_4l, *cxyz_space))
sigma_gg_4l_bsm = torch.sum(w_gg_4l_bsm, dim=0)  # c6 & cHbox

# 2l2v

w_gg_2l2v_sm = torch.tensor(events_gg_2l2v.weights.to_numpy())
sigma_gg_2l2v_sm = torch.sum(w_gg_2l2v_sm)

w_gg_2l2v_bsm = w_gg_2l2v_sm[:,None,None] * torch.tensor(eft.msq_eft_over_sm(coeffs_gg_2l2v, *cxyz_space))
sigma_gg_2l2v_bsm = torch.sum(w_gg_2l2v_bsm, dim=0)  # c6 & cHbox

/ptmp/mpp/taepa/higgs-offshell-interpretation/physics/hstar/eft.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  eft_coefficients = torch.tensor(eft_coefficients)


In [113]:
# # TEST: getting the ordering of events wrong should mess with everything below
# # w_pp_sm = mcfm.stack(events_gg_2l2v,events_qq_4l,events_gg_4l,events_qq_2l2v).weights.to_numpy()

w_pp_4l_sm = torch.tensor(mcfm.stack(events_gg_4l,events_qq_4l).weights.to_numpy())
sigma_pp_4l_sm = torch.sum(w_pp_4l_sm)
p_pp_4l_sm = w_pp_4l_sm / sigma_pp_4l_sm
nu_pp_4l_sm = sigma_pp_4l_sm * lumi

sigma_pp_4l_bsm = sigma_gg_4l_bsm + sigma_qq_4l_sm
nu_pp_4l_bsm = sigma_pp_4l_bsm * lumi

w_pp_2l2v_sm = torch.tensor(mcfm.stack(events_gg_2l2v,events_qq_2l2v).weights.to_numpy())
sigma_pp_2l2v_sm = torch.sum(w_pp_2l2v_sm)
p_pp_2l2v_sm = w_pp_2l2v_sm / sigma_pp_2l2v_sm
nu_pp_2l2v_sm = sigma_pp_2l2v_sm * lumi

sigma_pp_2l2v_bsm = sigma_gg_2l2v_bsm + sigma_qq_2l2v_sm
nu_pp_2l2v_bsm = sigma_pp_2l2v_bsm * lumi

In [114]:
# 4l

eft_mod_gg_4l = eft.Modifier(events=events_gg_4l)
w_gg_4l_bsm_asimov, _ = eft_mod_gg_4l.modify(*cxyz_asimov)
w_gg_4l_bsm_asimov = torch.tensor(w_gg_4l_bsm_asimov)

w_pp_4l_bsm_asimov = torch.cat((w_gg_4l_bsm_asimov, torch.tensor(events_qq_4l.weights.to_numpy())))
sigma_pp_4l_bsm_asimov = torch.sum(w_pp_4l_bsm_asimov, dim=0)
nu_pp_4l_bsm_asimov = sigma_pp_4l_bsm_asimov * lumi

# 2l2v

eft_mod_gg_2l2v = eft.Modifier(events=events_gg_2l2v)
w_gg_2l2v_bsm_asimov, _ = eft_mod_gg_2l2v.modify(*cxyz_asimov)
w_gg_2l2v_bsm_asimov = torch.tensor(w_gg_2l2v_bsm_asimov)

w_pp_2l2v_bsm_asimov = torch.cat((w_gg_2l2v_bsm_asimov, torch.tensor(events_qq_2l2v.weights.to_numpy())))
sigma_pp_2l2v_bsm_asimov = torch.sum(w_pp_2l2v_bsm_asimov)
nu_pp_2l2v_bsm_asimov = sigma_pp_2l2v_bsm_asimov * lumi

In [115]:
# Poisson term
def t_poisson_term(nu_bsm, nu_sm, nu_asimov):
    return  -2 * nu_asimov * (torch.log(nu_bsm) - torch.log(nu_sm)) + 2 * (nu_bsm - nu_sm) 

t_4l_rate = t_poisson_term(nu_pp_4l_bsm, nu_pp_4l_sm, nu_pp_4l_bsm_asimov)
t_2l2v_rate = t_poisson_term(nu_pp_2l2v_bsm, nu_pp_2l2v_sm, nu_pp_2l2v_bsm_asimov)
t_rate = t_4l_rate + t_2l2v_rate

In [116]:
# 4l

def pratio_pp_bsm(sigma_gg_sm, sigma_gg_bsm, sigma_qq_sm, lr, coeffs, cxyz):
    carl = sigma_qq_sm * torch.tensor(lr)[:,None,None]
    taylr = sigma_gg_sm * torch.tensor(eft.msq_eft_over_sm(coeffs, *cxyz))
    return (sigma_gg_sm+sigma_qq_sm)/(sigma_gg_bsm+sigma_qq_sm) * (taylr+carl)/(sigma_gg_sm+carl)

pratio_pp_4l_bsm =  pratio_pp_bsm(sigma_gg_4l_sm, sigma_gg_4l_bsm, sigma_qq_4l_sm, r_4l, coeffs_4l, cxyz_space)
pratio_pp_2l2v_bsm =  pratio_pp_bsm(sigma_gg_2l2v_sm, sigma_gg_2l2v_bsm, sigma_qq_2l2v_sm, r_2l2v, coeffs_2l2v, cxyz_space)

/tmp/ipykernel_2109190/1116108719.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  carl = sigma_qq_sm * torch.tensor(lr)[:,None,None]
/ptmp/mpp/taepa/higgs-offshell-interpretation/physics/hstar/eft.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  eft_coefficients = torch.tensor(eft_coefficients)


In [117]:
p_pp_4l_bsm = pratio_pp_4l_bsm * p_pp_4l_sm[:, None, None]
psum_pp_4l_bsm = torch.sum(p_pp_4l_bsm, dim=0) 
pratio_pp_4l_bsm = pratio_pp_4l_bsm / psum_pp_4l_bsm[None,:,:]

p_pp_2l2v_bsm = pratio_pp_2l2v_bsm * p_pp_2l2v_sm[:, None, None]
psum_pp_2l2v_bsm = torch.sum(p_pp_2l2v_bsm, dim=0) 
pratio_pp_2l2v_bsm = pratio_pp_2l2v_bsm / psum_pp_2l2v_bsm[None,:,:]

In [118]:
t_4l_shape = - 2 * torch.sum(lumi * torch.tensor(w_pp_4l_bsm_asimov)[:,None,None] * torch.log(pratio_pp_4l_bsm) , dim=0)
t_2l2v_shape = - 2 * torch.sum(lumi * torch.tensor(w_pp_2l2v_bsm_asimov)[:,None,None] * torch.log(pratio_pp_2l2v_bsm) , dim=0)
t_shape = t_4l_shape + t_2l2v_shape

/tmp/ipykernel_2109190/4158061025.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t_4l_shape = - 2 * torch.sum(lumi * torch.tensor(w_pp_4l_bsm_asimov)[:,None,None] * torch.log(pratio_pp_4l_bsm) , dim=0)
/tmp/ipykernel_2109190/4158061025.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t_2l2v_shape = - 2 * torch.sum(lumi * torch.tensor(w_pp_2l2v_bsm_asimov)[:,None,None] * torch.log(pratio_pp_2l2v_bsm) , dim=0)


In [119]:
t = t_rate + t_shape

In [120]:
i_cx_fit = torch.argmin(t) // cx_linspace[2]
i_cy_fit = torch.argmin(t)  % cy_linspace[2]

i_cx_fit_rate = torch.argmin(t_rate) // cx_linspace[2]
i_cy_fit_rate = torch.argmin(t_rate)  % cy_linspace[2]

t_min = t[i_cx_fit,i_cx_fit]
t_min_rate = t_rate[i_cx_fit_rate,i_cy_fit_rate]

X, Y = np.meshgrid(cx_space, cy_space)
Z = t.T
Z_rate = t_rate.T

cx_fit = cx_space[i_cx_fit]
cy_fit = cy_space[i_cy_fit]

In [121]:
# Functions
def mu_hh_cH_ctH(CH, CtH):
    return (
        1.0006080000000002 + 0.41783636507543154*CH + 
        0.0752601078185488*CH**2 + 0.28085172568843336*CtH + 
        0.053673806572758304*CH*CtH - 0.009570612033661833*CH**2*CtH + 
        0.05366821980400314*CtH**2 - 0.0021150770077951932*CH*CtH**2 + 
        0.00030426681994567096*CH**2*CtH**2 + 0.001827239604411044*CtH**3 - 
        0.0001899211252207785*CH*CtH**3 + 0.000036512477523666495*CtH**4
    )

def mu_hh_cH_cHG(CH, CHG):
    return (
        1.0006080000000002 + 0.41783636507543154*CH + 
        0.0752601078185488*CH**2 - 28.217798806017488*CHG - 
        9.355820574610702*CH*CHG + 1.9309976819615295*CH**2*CHG + 
        945.5252697479542*CHG**2 - 195.54992078191623*CH*CHG**2 + 
        13.931624998852588*CH**2*CHG**2 + 188.05434572117352*CHG**3 - 
        16.814389645769737*CH*CHG**3
    )

def mu_hh_ctH_cHG(CtH, CHG):
    return (
        1.0006080000000002
        - 28.21779880601748 * CHG
        + 945.5252697479541 * CHG**2
        + 188.05434572117352 * CHG**3
        + 0.28085172568843336 * CtH
        - 9.09813894164412 * CHG * CtH
        - 1.0073855372331442 * CHG**2 * CtH
        + 0.05366821980400314 * CtH**2
        - 0.2268838452243465 * CHG * CtH**2
        - 0.019958665669594654 * CHG**2 * CtH**2
        + 0.001827239604411044 * CtH**3
        + 0.00002835885257127466 * CHG * CtH**3
        + 0.000036512477523666495 * CtH**4
    )


In [ ]:
plt.figure(figsize=(5,4))

ax = plt.gca()

if cx_name=='ct' and cy_name=='cg':
    x1, y1 = np.min(cx_space), 0.00487036 * np.min(cx_space)
    x2, y2 = np.max(cx_space), 0.00487036 * np.max(cx_space)
    print(x1, x2)
    print(y1, y2)
    plt.axline((x1, y1), (x2, y2), linestyle=':', linewidth=1.5, color='tomato')

if cx_name=='c6' and cy_name=='ct':
    contours_hh = plt.contour(X, Y, mu_hh_cH_ctH(X, Y), levels=[-np.inf, 2], colors='forestgreen', linestyles=['-'], linewidths=[1.5])
if cx_name=='c6' and cy_name=='cg':
    contours_hh = plt.contour(X, Y, mu_hh_cH_cHG(X, Y), levels=[-np.inf, 2], colors='forestgreen', linestyles=['-'], linewidths=[1.5])
if cx_name=='ct' and cy_name=='cg':
    contours_hh = plt.contour(X, Y, mu_hh_ctH_cHG(X, Y), levels=[-np.inf, 2], colors='forestgreen', linestyles=['-'], linewidths=[1.5])

contours = plt.contour(X, Y, Z, levels=[t_min+1,t_min+4], colors='royalblue', linestyles=['--','-'], linewidths=[1.5,1.5])
contours_rate = plt.contour(X, Y, Z_rate, levels=[t_min_rate+1,t_min_rate+4], colors='grey', linestyles=['--','-'], linewidths=[1.5,1.5])

plt.scatter(cx_fit, cy_fit, marker='+', color='blue')

rate = Line2D([0], [0], color='grey', label='$\\mathrm{Rate}$')
nsbi = Line2D([0], [0], color='blue', label='$\\mathrm{NSBI}$')
sigma1 = Line2D([0], [0], color='black', linestyle='--', label='$1\\sigma$')
sigma2 = Line2D([0], [0], color='black', linestyle='-', label='$2\\sigma$')
bestfit = Line2D([0], [0], color='black', marker='+', linestyle='None', label='$\\textrm{SM}$')
empty = Line2D([0], [0], color='none', label='')
labels_hstar = [rate, nsbi, empty, sigma1, sigma2, bestfit]
leg_hstar = ax.legend(handles=labels_hstar, frameon=False, loc='upper right', ncols=2)
ax.add_artist(leg_hstar)  # This keeps the first legend

if cx_name=='c6':
    labels_hh = [
        Line2D([0],[0],color='forestgreen',linestyle='-',label='$\\mu_{hh} < 2$'),
    ]
    plt.legend(frameon=False, handles=labels_hh, loc='lower right')
else:
    labels_h125 = [
        Line2D([0],[0],color='tomato',linestyle=':',label='$\\textrm{On-shell degeneracy}$'),
        Line2D([0],[0],color='forestgreen',linestyle='-',label='$\\mu_{hh} < 2$'),
    ]
    plt.legend(frameon=False, handles=labels_h125, loc='lower right')

plt.text(0.04 ,0.96, '$pp \\rightarrow ZZ$', transform=ax.transAxes, ha='left', va='top', fontsize=12)
plt.text(0.04 ,0.88, '$\\sqrt{s} = 14\\,\\mathrm{TeV},\\  3\\, \\mathrm{ab}^{-1}$', transform=ax.transAxes, ha='left', va='top', fontsize=12)

plt.xlabel(cx_label, fontsize=12)
plt.ylabel(cy_label, fontsize=12)

plt.xlim(cx_min, cx_max)
plt.ylim(cy_min, cy_max)

plt.tick_params(axis='both', labelsize=12)

plt.tight_layout()
plt.savefig(f'nll_pp_zz_{cx_name}_{cy_name}_contours.pdf', bbox_inches='tight')

-19.65916797009516 19.65916797009516
-0.09574722531483267 0.09574722531483267


In [123]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import TwoSlopeNorm

# Your Z array (already defined)
Z = torch.pow(1. / psum_pp_4l_bsm, sample_size)

# Create a normalization centered at 1.0
norm = TwoSlopeNorm(vmin=1-torch.max(torch.abs(Z-1))*1.1, vcenter=1.0, vmax=1+torch.max(torch.abs(Z-1))*1.1)

# Plot with imshow
plt.figure(figsize=(5, 4))
plt.imshow(
    Z,
    origin='lower',
    aspect='auto',
    extent=[
        cx_space[0], cx_space[-1],
        cy_space[0], cy_space[-1]
    ],
    # cmap='seismic', 
    # cmap='coolwarm',
    cmap='bwr', 
    norm=norm
)
plt.colorbar(label='$\\mathrm{Probability\\ ratio\\ calibration}$')
plt.xlabel(cx_label)
plt.ylabel(cy_label)

plt.tight_layout()
plt.savefig(f'pratio_calibration_{cx_name}_{cy_name}.pdf', bbox_inches='tight')


In [124]:
import gc
torch.cuda.empty_cache()
gc.collect()

250